In [1]:
# --- Notebook Test Engine: injected setup (auto-generated) ---
import os as _os
for _k in ('AWS_ACCESS_KEY_ID','AWS_SECRET_ACCESS_KEY','AWS_SESSION_TOKEN','AWS_CREDENTIAL_EXPIRATION'):
    _os.environ.pop(_k, None)
_os.environ.setdefault('AWS_DEFAULT_REGION', 'us-west-2')
_os.environ.setdefault('AWS_REGION', 'us-west-2')
_nte_role = 'arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole'
try:
    from sagemaker.core.helper import session_helper as _sh
    _sh.get_execution_role = lambda *a, **k: _nte_role
except Exception:
    pass
try:
    _nte_cf = _os.environ.get('AWS_SHARED_CREDENTIALS_FILE')
    if _nte_cf and _os.path.exists(_nte_cf):
        import configparser as _cfp, datetime as _dt
        import boto3 as _b3, botocore.session as _bcs
        from botocore.credentials import RefreshableCredentials as _RC
        _nte_prof = _os.environ.get('AWS_PROFILE', 'default')
        def _nte_load():
            _cp = _cfp.ConfigParser(); _cp.read(_nte_cf)
            _s = _cp[_nte_prof]
            _tok = _s.get('aws_session_token')
            if not _tok:
                raise RuntimeError('nte: no session token in creds file')
            return {'access_key': _s['aws_access_key_id'],
                    'secret_key': _s['aws_secret_access_key'],
                    'token': _tok,
                    'expiry_time': (_dt.datetime.now(_dt.timezone.utc)
                                    + _dt.timedelta(minutes=9)).isoformat()}
        _nte_creds = _RC.create_from_metadata(metadata=_nte_load(),
                                              refresh_using=_nte_load,
                                              method='nte-shared-file')
        def _nte_get_credentials(self, *a, **k):
            # Honor sessions that were handed explicit credentials
            # (e.g. notebooks that assume_role into other accounts):
            # clobbering them would silently run every cross-account
            # client as the ambient identity. Bare/SDK-created sessions
            # (no explicit creds) still get the refreshable shared-file
            # creds so long-running waits survive credential rotation.
            _ex = getattr(self, '_credentials', None)
            if _ex is not None:
                return _ex
            return _nte_creds
        _bcs.Session.get_credentials = _nte_get_credentials
        _b3.setup_default_session(botocore_session=_bcs.get_session())
        print('nte: refreshable shared-file creds installed')
except Exception as _e:
    print('nte: refreshable-creds shim skipped:', _e)
try:
    from sagemaker.core.resources import ModelPackageGroup as _NteMPG
    _nte_mpg_create = _NteMPG.create
    def _nte_mpg_get_or_create(*_a, **_k):
        try:
            return _nte_mpg_create(*_a, **_k)
        except Exception as _e2:
            if 'already exists' in str(_e2).lower():
                _name = _k.get('model_package_group_name') or (_a[0] if _a else None)
                return _NteMPG.get(_name)
            raise
    _NteMPG.create = staticmethod(_nte_mpg_get_or_create)
    print('nte: ModelPackageGroup.create is now get-or-create')
except Exception as _e:
    print('nte: ModelPackageGroup idempotency shim skipped:', _e)


nte: ModelPackageGroup.create is now get-or-create


# Direct Preference Optimization (DPO) Training with SageMaker

This notebook demonstrates how to use the **DPOTrainer** to fine-tune large language models using Direct Preference Optimization (DPO). DPO is a technique that trains models to align with human preferences by learning from preference data without requiring a separate reward model.

## What is DPO?

Direct Preference Optimization (DPO) is a method for training language models to follow human preferences. Unlike traditional RLHF (Reinforcement Learning from Human Feedback), DPO directly optimizes the model using preference pairs without needing a reward model.

**Key Benefits:**
- Simpler than RLHF - no reward model required
- More stable training process
- Direct optimization on preference data
- Works with LoRA for efficient fine-tuning

## Workflow Overview

1. **Prepare Preference Dataset**: Upload preference data in JSONL format
2. **Register Dataset**: Create a SageMaker AI Registry dataset
3. **Configure DPO Trainer**: Set up model, training parameters, and resources
4. **Execute Training**: Run the DPO fine-tuning job
5. **Track Results**: Monitor training with MLflow integration

## Step 1: Prepare and Register Preference Dataset

DPO requires preference data in a specific format where each example contains:
- **prompt**: The input text
- **chosen**: The preferred response
- **rejected**: The less preferred response

The dataset should be in JSONL format with each line containing one preference example.

In [2]:
from sagemaker.ai_registry.dataset import DataSet
from sagemaker.ai_registry.dataset_utils import CustomizationTechnique


# Register dataset in SageMaker AI Registry
# This creates a versioned dataset that can be referenced by ARN
# Provide a source (it can be local file path or S3 URL)
dataset = DataSet.create(
    name="demo-6",
    source="s3://notebook-test-engine-ds-674622101542-usw2/datasets/sample-dpo-preference-208.jsonl"
)

print(f"Dataset ARN: {dataset.arn}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Fetched defaults config from location: /opt/ml/processing/input/sm_config.yaml


[07/20/26 18:11:51] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=15965170;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=15965171;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/20/26 18:11:52] INFO     Cannot simulate policies for                                  ]8;id=15965178;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=15965179;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

                    WARNING  Could not verify permissions for role                         ]8;id=15965185;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=15965186;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'training' (see                                                       
                             IamRoleResolver().get_required_actions('training')) or create                         
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='training')                         
                             .                                                                                     

...

[07/20/26 18:11:57] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=15965223;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=15965224;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

/usr/local/lib/python3.12/dist-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=15965229;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=15965230;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

Final Resource Status: Available

Dataset ARN: arn:aws:sagemaker:us-west-2:674622101542:hub-content/R8VBAD7SJ9M4M5ATDE0MIPSEJU3M1TEH5UCUNB65JMGNEP87RHFG/DataSet/demo-6/8.0.0


##### Create a Model Package group (if not already exists)

In [3]:
from sagemaker.core.resources import ModelPackage, ModelPackageGroup

model_package_group=ModelPackageGroup.create(model_package_group_name="test-model-package-group")

[07/20/26 18:11:58] INFO     Creating model_package_group resource.                              ]8;id=15965237;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=15965238;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#23250\23250]8;;\

                    WARNING  No region provided. Using default region.                                 ]8;id=15965245;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=15965246;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py#361\361]8;;\

                    INFO     Runs on sagemaker prod, region:us-west-2                                  ]8;id=15965252;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=15965253;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py#375\375]8;;\

## Step 2: Configure and Execute DPO Training

The **DPOTrainer** provides a high-level interface for DPO fine-tuning with the following key features:

### Key Parameters:
**Required Parameters** 

* `model`: base_model id on Sagemaker Hubcontent that is available to finetune (or) ModelPackage artifacts

**Optional Parameters**
* `training_type`: Choose from TrainingType Enum(sagemaker.modules.train.common) either LORA OR FULL.
* `model_package_group`: ModelPackage group name or ModelPackageGroup object. This parameter is mandatory when a base model ID is provided, but optional when a model package is provided.
* `mlflow_resource_arn`: MLFlow app ARN to track the training job
* `mlflow_experiment_name`: MLFlow app experiment name(str)
* `mlflow_run_name`: MLFlow app run name(str)
* `training_dataset`: Training Dataset - should be a Dataset ARN or Dataset object (Please note training dataset is required for a training job to run, can be either provided via Trainer or .train())
* `validation_dataset`: Validation Dataset - should be a Dataset ARN or Dataset object
* `s3_output_path`: S3 path for the trained model artifacts

### Training Features:
- **Serverless Training**: Automatically managed compute resources
- **LoRA Integration**: Parameter-efficient fine-tuning
- **MLflow Tracking**: Automatic experiment and metrics logging
- **Model Versioning**: Automatic model package creation

#### Reference 
Refer this doc for other models that support Model Customization: 
https://docs.aws.amazon.com/bedrock/latest/userguide/custom-model-supported.html

In [4]:
import random
#! ada credentials update --provider=isengard --account=<> --role=Admin --profile=default --once
#! aws configure set region  us-west-2

from sagemaker.train.dpo_trainer import DPOTrainer
from sagemaker.train.common import TrainingType



In [5]:
# Create DPOTrainer instance with comprehensive configuration
trainer = DPOTrainer(
    # Base model from SageMaker Hub
    model="meta-textgeneration-llama-3-2-1b-instruct",
    
    # Use LoRA for efficient fine-tuning
    training_type=TrainingType.LORA,
    
    # Model versioning and storage
    model_package_group=model_package_group, # or use an existing model package group arn
        
    # Training data (from Step 1)
    training_dataset=dataset.arn,
    
    # Output configuration
    s3_output_path="s3://notebook-test-engine-ds-674622101542-usw2/output/",

    
    # Unique job name
    base_job_name=f"dpo-job-{random.randint(1, 1000)}",
    accept_eula=True
)

# Customize training hyperparameters
# DPO-specific parameters are automatically loaded from the model's recipe
trainer.hyperparameters.max_epochs = 1  # Quick training for demo

print("Starting DPO training job...")
print(f"Job name: {trainer.base_job_name}")
print(f"Base model: {trainer._model_name}")

# Execute training with monitoring
training_job = trainer.train(wait=True)

print(f"Training completed! Job ARN: {training_job.training_job_arn}")

╭────────────────────────────────── Training Job Status ───────────────────────────────────╮
│  TrainingJob Name      dpo-job-299-20260720181201                                        │
│  TrainingJob ARN       arn:aws:sagemaker:us-west-2:674622101542:training-job/dpo-job-29  │
│                        9-20260720181201                                                  │
│  MLflow Experiment     test-model-package-group                                          │
│  Links                 ]8;id=15968677;https://us-west-2.console.aws.amazon.com/sagemaker/home?region=us-west-2#/jobs/dpo-job-299-20260720181201\🔗 Training Job (Console)]8;;\                                         │
│                        ]8;id=15968682;https://us-west-2.console.aws.amazon.com/cloudwatch/home?region=us-west-2#logsV2:log-groups/log-group/$252Faws$252Fsagemaker$252FTrainingJobs$3FlogStreamNameFilter$3Ddpo-job-299-20260720181201\🔗 CloudWatch Logs]8;;\ | ]8;id=15968683;https://app-3THB6MFIDE4F.mlflow.sagemaker.us-west-2.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6Ik1TTTM2RCIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNE95MFdoU2diTkVSS0JBczFqMlFrTmg2bjRsM2g1ZHp4QXp2aTVFbEpvT3NBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVGclkzWkpabHBCU25kcFkwZDJOMU4zYVZkR1RYbDRVRU4zZURWak5qSnBSVEUwYXpJMVNIbGhWWFZRWmxoSk0yRjJTRFowWmpOcE5HRmhSV2R4UWtjNWR6MDlBQUVBQjJGM2N5MXJiWE1BUzJGeWJqcGhkM002YTIxek9uVnpMWGRsYzNRdE1qbzVOell4T1RNeU5UUTFOemM2YTJWNUwyVmhNbUZrTUdSa0xXRmtNV0l0TkROalppMDRaak0xTFRNMU5UWmlZVE5pWm1NeE1nQzRBUUlCQUhqRHJwaCtQMVNKcHUzZUl2YnVXUi94bGQ2THMxNjhEU3JZbXo5akp0dlBUQUcwMld2MDRCekV2cU9CMHFwMTVTVFBBQUFBZmpCOEJna3Foa2lHOXcwQkJ3YWdiekJ0QWdFQU1HZ0dDU3FHU0liM0RRRUhBVEFlQmdsZ2hrZ0JaUU1FQVM0d0VRUU1iQnNwb25yRWwraGFFNkgrQWdFUWdEc1ZSVW9jWTM2Mk1ZcGJLeDhtU2JOblMyZTk5a1BCMzRrMElUSW9rWWZLRytpb0R0S05ZTEx3cmI3bWpNN0IxS1duTmJETTliZzd5YmpqakFJQUFCQUE3VWxoTXZmL1NDK2E0bVg4bzVXdkgrNHJUZ0xHeE9TSVo5cCtuTkFmZHBuVDJLZDRpUmhFSjBhak5OOVZYTnFyLy8vLy93QUFBQUVBQUFBQUFBQUFBQUFBQUFFQUFBUVhZWit4b1ZkbzR4NWVBTTQ0QVdvN0lqaXg2d3pzblRHS0dMV2Y0T2tzbjZIU2tUNFkzdmRyQ2wyL1FoLzJEQjF6VEtyeksyWFI0UlNpbk1IUkc2cjJtaFQrTnNva2FaenlZcnhGNTRRRlpXSmpTSGVMREt3cGJ1VVJWc3c3dXdVcEYrcnpEa0R1NlJUTHkrTXhqTC9uL1B3Q1p4RDhUZDdFWFVxdDdweUJRamxwZzViSVFsTnlBMnc2U1hsejVWZjY4ZVMybU11aFZCQzZIeHNSSVNXenBpN0pEazJGU2szRnpVaVA3MlJoK2xwTTAwQ2tWMC9QSlhKYnNkeXdJc2hPWUQ5TGRzZDFsQ054blZreDB2WUwrRmJicFFKbDFlZ3RGZEVRbi8zN1JHUXlrc204OEQrT1ZvTkxhc2Z6SVNnWnFma2ZrTnJmNVhldjNjVU1OUXpLRkxlbXhxc2xCRUJYQlQvbThra3pHbWlpeWRmTFRDTi9NYnRuSElFNzhxS1VOZFlXMEQwRkh1MDZoNEpzRFpBbnU4SzJKWkpQbEg5UStic2kyeUFhY2ZQQjQ4Ym5EWWxrMnJxTGMvaUlUbm1lVHRSd2cvZjdKbk5xU3l6OUZLZTdZTHMrcnV5WVVrai91eFJaaUx0cVFaVnZVQUh3b0FLS3JFWmF6bDkwUWpyTjhBSUVySHBGdUFmSldpWUFsd0h2d2lvT2FmclE1L1N6MFdoZ1pHR0thejd2SVVQdWMyMWJSNmRJeXl2eXEvaHRwdXBPODdKNkY2dXh2NTk2MzErNWJNK2Yrd2YybzQ5QXZ1VzNoTzY2Q0gvaTFaYzVYUWFtZUJ2YUFCT3NQQWtkWHVWakJoVHlQdWpjQWtCcnZRWHEzcVVGLzRRUXdwNkROMFlPNXV2ZjdpZERZLzk0S0gySnYveWhrTHg3VjdwVStGZ1RuVVNPVDJVRWxBRTI3Y0JvMDUyWFFMRFhSd3piakpKdElPR1psV2l4U2lad2ZOMEYyY1BCaDltM0g5Z0RwTkE2UHIxQlErdTVPRU1pOVpCU09Jc1IyeGN4R2tNZHg1dmVUcFlNT2VPeWVNdUhQYmJ6cFJzRzdUUjN4T2J6OFJkU0NWUTRqdGM1QmYxMURzTjBXVmdpaDBrUk9sc1EyZ2hSditkcWQ5TUhPQ2RuZ2RKdThRdXZhczRFaFBGRU45bjFqTzQvVHBsSlgyNmI3UlUyU3d3b1dNLzNmZEQxb3dNdWx4L293YlBkNmZPenlMWmVtcWRoMGtsVSsxZ2RkdVk1NjhsUVkvS2RVWTBVbXNKVEtsRlV5VjE1U3ZSNVB0VFlDaWJmNVZSZDh5bVh6SFI1clNXcUMvMjc1eExLY0V6U09GSUc2bzk5WTJnVlRkRTIzMUREWHIyeCtlSGlTMkJwSU9IbU1yaVN6M1gwQmxXOGlnMHNXY0EvQXFES1l2ZG9nVDVGVmZRaW9oQmUwVWN2anhCUnBTZ0JNS2dZK05VVEZFVWlWVllLUTVYRUJkTzQrWHRJWFp0ZkJjcCtIYXJwTnAvMGkrQ1FzaFJDRHNsWE9NMFdsOUNONmFCbk8ycnJyVitXNTlIb01qRGlzNk5CcFNzMzRWVFJVYW1kM3BXWGV5OTU1SUpnc0laQm5xMG5pdGNhSjZKc3dmSmJPUFRZQTJFNGF2YXNVQ3RLQVE5djJxdFUzTWlnSVdXVVZvQkJ6YkZFZjJ5Rm9HSmI0NDdVUTF1MFFCT1ZieW9VWHNzUmRldTRuMENDVDg0dnF2YkxvZ21YMkhkK2pSUGk1dEhCb2J5TWNvTkRvNElWeWp5MmlQV0V0M3ZwU203WUF2b01hRWE2cGpUdVJZRlFSNFBJRXgrampSTWRvRGNuM3Q2YVNUSEwzUUJuTUdVQ01RREJUaFVta0pNUi9Db29GeG9mdUNRT044YjlERWdnbzEyNzRabE5iQTdDOTN1b0VYYzFncVdIWEpodUVLWWU5eXNDTUZPWEU2M3J6RFkzQy9HdjJXd1l3ZzY2QTR

Training completed! Job ARN: arn:aws:sagemaker:us-west-2:674622101542:training-job/dpo-job-299-20260720181201


In [6]:
import json
import re
from pprint import pprint
from sagemaker.core.utils.utils import Unassigned

def pretty_print(obj):
    def parse_unassigned(item):
        if isinstance(item, Unassigned):
            return None
        if isinstance(item, dict):
            return {k: parse_unassigned(v) for k, v in item.items() if parse_unassigned(v) is not None}
        if isinstance(item, list):
            return [parse_unassigned(x) for x in item if parse_unassigned(x) is not None]
        if isinstance(item, str) and "Unassigned object" in item:
            pairs = re.findall(r"(\w+)=([^<][^=]*?)(?=\s+\w+=|$)", item)
            result = {k: v.strip("'\"") for k, v in pairs}
            return result if result else None
        return item

    cleaned = parse_unassigned(obj.__dict__ if hasattr(obj, '__dict__') else obj)
    print(json.dumps(cleaned, indent=2, default=str))
pretty_print(training_job)

{
  "training_job_name": "dpo-job-299-20260720181201",
  "training_job_arn": "arn:aws:sagemaker:us-west-2:674622101542:training-job/dpo-job-299-20260720181201",
  "model_artifacts": "s3_model_artifacts='s3://notebook-test-engine-ds-674622101542-usw2/output/dpo-job-299-20260720181201/output/model'",
  "training_job_status": "Completed",
  "secondary_status": "Completed",
  "hyper_parameters": {
    "adam_beta": "0.01",
    "dataset_max_len": "4096",
    "global_batch_size": "16",
    "gradient_clipping": "True",
    "gradient_clipping_threshold": "1.0",
    "learning_rate": "0.0001",
    "logging_steps": "1",
    "lora_alpha": "32",
    "lora_dropout": "0.05",
    "lora_rank": "16",
    "lr_scheduler": "cosine",
    "lr_warmup_steps_ratio": "0.1",
    "max_epochs": "1",
    "merge_weights": "True",
    "model_name_or_path": "meta-llama/Llama-3.2-1B-Instruct",
    "name": "example-name-2qffj",
    "seed": "42",
    "temperature": "0.6",
    "train_val_split_ratio": "0.9",
    "weight_dec

In [7]:
# Print the training job object

import json
from sagemaker.core.utils.utils import Unassigned
from sagemaker.core.resources import TrainingJob
import pprint
response = TrainingJob.get(training_job_name=training_job.training_job_name)

import json
import re
from sagemaker.core.utils.utils import Unassigned

# Usage
pretty_print(response)



{
  "training_job_name": "dpo-job-299-20260720181201",
  "training_job_arn": "arn:aws:sagemaker:us-west-2:674622101542:training-job/dpo-job-299-20260720181201",
  "model_artifacts": "s3_model_artifacts='s3://notebook-test-engine-ds-674622101542-usw2/output/dpo-job-299-20260720181201/output/model'",
  "training_job_status": "Completed",
  "secondary_status": "Completed",
  "hyper_parameters": {
    "adam_beta": "0.01",
    "dataset_max_len": "4096",
    "global_batch_size": "16",
    "gradient_clipping": "True",
    "gradient_clipping_threshold": "1.0",
    "learning_rate": "0.0001",
    "logging_steps": "1",
    "lora_alpha": "32",
    "lora_dropout": "0.05",
    "lora_rank": "16",
    "lr_scheduler": "cosine",
    "lr_warmup_steps_ratio": "0.1",
    "max_epochs": "1",
    "merge_weights": "True",
    "model_name_or_path": "meta-llama/Llama-3.2-1B-Instruct",
    "name": "example-name-2qffj",
    "seed": "42",
    "temperature": "0.6",
    "train_val_split_ratio": "0.9",
    "weight_dec

## Next Steps

After training completes, you can:

1. **Deploy the Model**: Use `ModelBuilder` to deploy the fine-tuned model
2. **Evaluate Performance**: Compare responses from base vs fine-tuned model
3. **Monitor Metrics**: Review training metrics in MLflow
4. **Iterate**: Adjust hyperparameters and retrain if needed

### Example Deployment:
```python
from sagemaker.serve import ModelBuilder

# Deploy the fine-tuned model
model_builder = ModelBuilder(model=training_job)
model_builder.build(role_arn="arn:aws:iam::account:role/SageMakerRole")
endpoint = model_builder.deploy(endpoint_name="dpo-finetuned-llama")
```

The fine-tuned model will now generate responses that better align with the preferences in your training data.

## Iterative Training (Resume from Checkpoint)

Resume DPO training from a previously trained model package. Pass the model package directly — the service uses `SourceModelPackageArn` to resume from that checkpoint.

#### Fetch the Model Package ARN from a previous Training Job

After a serverless training job completes, it produces an `OutputModelPackageArn` that can be used as the base model for iterative (continued) fine-tuning. Here's how to retrieve it:

In [8]:
from sagemaker.core.resources import TrainingJob

# Get the completed training job
previous_job = TrainingJob.get(training_job_name=training_job.training_job_name)

# The output model package ARN is available on completed serverless training jobs
output_model_package_arn = previous_job.output_model_package_arn
print(f"Output Model Package ARN: {output_model_package_arn}")

# Use this ARN to get the ModelPackage for iterative training
from sagemaker.core.resources import ModelPackage

model_package = ModelPackage.get(model_package_name=output_model_package_arn)
pretty_print(model_package)

Output Model Package ARN: arn:aws:sagemaker:us-west-2:674622101542:model-package/test-model-package-group/1


{
  "model_package_group_name": "test-model-package-group",
  "model_package_version": 1,
  "model_package_registration_type": "Logged",
  "model_package_arn": "arn:aws:sagemaker:us-west-2:674622101542:model-package/test-model-package-group/1",
  "creation_time": "2026-07-20 18:22:46.260000+00:00",
  "inference_specification": "containers=[ContainersItem(model_data_url=None, image=None, nearest_model_name=None, model_data_source=ModelDataSource(s3_data_source=S3ModelDataSource(s3_data_type='S3Prefix', compression_type='None', s3_uri='s3://notebook-test-engine-ds-674622101542-usw2/output/dpo-job-299-20260720181201/output/model/', model_access_config=Unassigned(), hub_access_config=Unassigned(), manifest_s3_uri=Unassigned(), e_tag=Unassigned(), manifest_etag=Unassigned())), is_checkpoint=False, base_model=BaseModel(hub_content_name='meta-textgeneration-llama-3-2-1b-instruct', hub_content_version='2.9.0', recipe_name='llmft_llama3_2_1b_instruct_seq4k_gpu_dpo'))]",
  "model_package_status"

In [9]:
from sagemaker.core.resources import ModelPackageGroup
# Iterative DPO training from a model package
# Pass the model package from a previous training run
trainer_iterative = DPOTrainer(
    model=model_package,  # ModelPackage from a previous training run
    training_type=TrainingType.LORA,
    model_package_group=ModelPackageGroup.create(model_package_group_name="my-iterative-models"),
    training_dataset=dataset.arn,
    s3_output_path="s3://notebook-test-engine-ds-674622101542-usw2/iterative-output/",
    accept_eula=True,
)

training_job = trainer_iterative.train(wait=False)
print(f"Iterative DPO job submitted: {training_job.training_job_name}")

[07/20/26 18:22:50] INFO     Creating model_package_group resource.                              ]8;id=15968688;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=15968689;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#23250\23250]8;;\

                    INFO     SageMaker session not provided. Using default Session.                  ]8;id=15968694;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=15968695;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

[07/20/26 18:22:51] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=15968700;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=15968701;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/20/26 18:22:52] INFO     SageMaker session not provided. Using default Session.                  ]8;id=15968706;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=15968707;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

                    INFO     Cannot simulate policies for                                  ]8;id=15968712;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=15968713;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

...

                    INFO     SageMaker session not provided. Using default Session.                  ]8;id=15968754;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=15968755;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

                    INFO     Auto-detecting whether dataset is multimodal:                        ]8;id=15968760;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/data_utils.py\data_utils.py]8;;\:]8;id=15968761;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/data_utils.py#140\140]8;;\
                             arn:aws:sagemaker:us-west-2:674622101542:hub-content/R8VBAD7SJ9M4M5A                  
                             TDE0MIPSEJU3M1TEH5UCUNB65JMGNEP87RHFG/DataSet/demo-6/8.0.0                            

                    WARNING  Failed to check multimodal data from                                 ]8;id=15968766;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/data_utils.py\data_utils.py]8;;\:]8;id=15968767;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/common_utils/data_utils.py#170\170]8;;\
                             arn:aws:sagemaker:us-west-2:674622101542:hub-content/R8VBAD7SJ9M4M5A                  
                             TDE0MIPSEJU3M1TEH5UCUNB65JMGNEP87RHFG/DataSet/demo-6/8.0.0: File                      
                             must have .jsonl extension:                                                           
                             arn:aws:sagemaker:us-west-2:674622101542:hub-content/R8VBAD7SJ9M4M5A                  
                             TDE0MIPSEJU3M1TEH5UCUNB65JMGNEP87RHFG/DataSet/demo-6/8.0.0.                           
                             Defaulting to text-only.                                                              

                    INFO     Creating training_job resource.                                     ]8;id=15968772;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=15968773;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#31239\31239]8;;\

Iterative DPO job submitted: meta-textgeneration-llama-3-2-1b-instruct-dpo-20260720182252
